# BraTS post processing

In [1]:
import os

# Path to nnUNet labels for the test split (set up by the user outside the repo)
DIR_DATA  = "../../../../../data/nnUNet_raw_data/Task182_BraTS2020_80_20/labelsTs/"
DIR_PREDS = "./datasets/"

gt_files = sorted(f for f in os.listdir(DIR_DATA)  if f.endswith('.nii.gz'))
pred_npz  = sorted(f for f in os.listdir(DIR_PREDS) if f.endswith('.npz'))
case_ids  = [f.replace('.npz', '') for f in pred_npz]

print(f"Ground truth files : {len(gt_files)}")
print(f"Prediction cases   : {len(case_ids)}")
print(f"First case id      : {case_ids[0]}")

Ground truth files : 73
Prediction cases   : 73
First case id      : BraTS20_Training_002


## Whole Tumour (WT) extraction

The model is a multi-region nnUNet ensemble trained on BraTS 2020. Each `.npz` file
contains a `softmax` array of shape `(3, Z, Y, X)` in nnUNet's internal axis order
(slice-first), with one sigmoid channel per region:

| Channel | Region |
|---------|--------|
| 0 | **Whole Tumour (WT)** — all non-background voxels |
| 1 | Tumour Core (TC) |
| 2 | Enhancing Tumour (ET) |

The corresponding `.nii.gz` file stores the post-processed multi-class prediction in
NIfTI axis order `(X, Y, Z)`. Label `> 0` recovers the WT binary mask.

Ground-truth labels follow the original BraTS convention (0 = background, 1 = NCR/NET,
2 = oedema, 4 = enhancing); WT ground truth = `gt > 0`.

Each 3-D volume (240 × 240 × 155 voxels) is split into 155 axial slices. Every slice
becomes one independent sample in the output `brats.npz`.

In [2]:
import numpy as np
import nibabel as nib
from tqdm import tqdm

all_p_hat, all_y_hat, all_y = [], [], []

for case_id in tqdm(case_ids, desc="Processing cases"):
    # --- softmax: (3, Z, Y, X) nnUNet axis order ---
    softmax = np.load(os.path.join(DIR_PREDS, case_id + '.npz'))['softmax'].astype(np.float32)
    # Channel 0 = WT probability; transpose to NIfTI (X, Y, Z) order
    p_hat_vol = softmax[0].transpose(2, 1, 0)          # (240, 240, 155)

    # --- binarised prediction: (X, Y, Z), labels 0-3; WT = any label > 0 ---
    pred_arr = np.array(
        nib.load(os.path.join(DIR_PREDS, case_id + '.nii.gz')).dataobj,
        dtype=np.uint8,
    )
    y_hat_vol = (pred_arr > 0).astype(np.float32)       # (240, 240, 155)

    # --- ground truth: (X, Y, Z), BraTS labels {0,1,2,4}; WT = any label > 0 ---
    gt_arr = np.array(
        nib.load(os.path.join(DIR_DATA, case_id + '.nii.gz')).dataobj,
        dtype=np.uint8,
    )
    y_vol = (gt_arr > 0).astype(np.float32)             # (240, 240, 155)

    # --- split into axial slices along Z (last axis) ---
    n_slices = p_hat_vol.shape[2]                        # 155
    for z in range(n_slices):
        all_p_hat.append(p_hat_vol[:, :, z][np.newaxis])   # (1, 240, 240)
        all_y_hat.append(y_hat_vol[:, :, z][np.newaxis])   # (1, 240, 240)
        all_y.append(y_vol[:, :, z][np.newaxis])            # (1, 240, 240)

p_hat = np.stack(all_p_hat)   # (N, 1, 240, 240)
y_hat = np.stack(all_y_hat)   # (N, 1, 240, 240)
y     = np.stack(all_y)       # (N, 1, 240, 240)

print(f"Total slices : {len(all_p_hat)}")
print(f"p_hat shape  : {p_hat.shape}")
print(f"y_hat shape  : {y_hat.shape}")
print(f"y shape      : {y.shape}")

Processing cases: 100%|██████████| 73/73 [00:16<00:00,  4.43it/s]


Total slices : 11315
p_hat shape  : (11315, 1, 240, 240)
y_hat shape  : (11315, 1, 240, 240)
y shape      : (11315, 1, 240, 240)


In [3]:
output_path = os.path.join('..', '..', 'medical-imaging', 'data', 'brats.npz')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
np.savez_compressed(output_path, p_hat=p_hat, y_hat=y_hat, y=y)
print(f"Saved → {output_path}")

Saved → ../../medical-imaging/data/brats.npz


## Sanity check

Verify the saved file and report dataset statistics.

In [4]:
data = np.load(output_path)
p_hat_v, y_hat_v, y_v = data['p_hat'], data['y_hat'], data['y']

tumor_slices = np.any(y_v > 0, axis=(-3, -2, -1))
print(f"Slices total         : {len(y_v)}")
print(f"Slices with tumour   : {tumor_slices.sum()} ({100*tumor_slices.mean():.1f} %)")
print(f"p_hat range          : [{p_hat_v.min():.4f}, {p_hat_v.max():.4f}]")

# Dice score on non-empty slices (using y_hat vs y, as in RC_curves.ipynb)
def dice(a, b):
    inter = (a * b).sum()
    return 2 * inter / (a.sum() + b.sum() + 1e-8)

non_empty = y_v[tumor_slices]
y_hat_ne  = y_hat_v[tumor_slices]
dice_scores = [dice(y_hat_ne[i], non_empty[i]) for i in range(len(non_empty))]
print(f"Mean Dice (WT, non-empty slices) : {np.mean(dice_scores):.4f}")

Slices total         : 11315
Slices with tumour   : 4831 (42.7 %)
p_hat range          : [0.0000, 1.0000]
Mean Dice (WT, non-empty slices) : 0.8314
